<a href="https://colab.research.google.com/github/Orliluq/ONE-AI-FOR-TECH-RAG/blob/main/Inteligencia_de_Datos_y_RAG_Avanzado_ONE_AI_FOR_TECH.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain langchain-google-genai google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.9 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

In [3]:
import os

# Suponiendo que tienes una Clave de API almacenada en una variable de entorno
api_key = os.getenv('GEMINI_API_KEY')

# Usa la Clave de API para acceder a un servicio
print(f"Usando la Clave de API: {api_key}")

Usando la Clave de API: None


In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    google_api_key=GEMINI_API_KEY
)

In [5]:
respuesta = llm.invoke("¿Qué es el RAG en Inteligencia Artificial?")

In [6]:
respuesta.content

'**RAG** significa **Retrieval-Augmented Generation** (Generación Aumentada por Recuperación). Es una técnica en Inteligencia Artificial que mejora la capacidad de los Modelos de Lenguaje Grandes (LLMs, por sus siglas en inglés, como GPT-4) para generar respuestas más precisas, relevantes y actualizadas, al permitirles acceder y utilizar información externa y específica en tiempo real.\n\nEn esencia, RAG combina la potencia de los LLMs con la capacidad de buscar y recuperar información de una base de datos o un conjunto de documentos.\n\n### ¿Por qué es necesario el RAG? (El Problema)\n\nLos LLMs, a pesar de su impresionante capacidad, tienen algunas limitaciones inherentes:\n\n1.  **Conocimiento Estático y Desactualizado:** Su conocimiento se limita a los datos con los que fueron entrenados. No tienen acceso a información reciente o a eventos posteriores a su fecha de corte de entrenamiento.\n2.  **Alucinaciones:** A veces, los LLMs pueden "inventar" hechos o generar información incor

In [7]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

# Asumiendo que GEMINI_API_KEY ya ha sido definida en una celda anterior.
# Creando una instancia del LLM de Google
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", # Cambiado de "gemini-pro" a "gemini-2.5-flash"
    temperature=0,
    google_api_key=GEMINI_API_KEY
)

# Usando el LLM para generar texto
prompt_text = "Explica qué es inteligencia artificial."
response = llm.invoke([HumanMessage(content=prompt_text)]) # Wrap HumanMessage in a list

# Exibiendo la respuesta generada por el LLM
print(response.content)

La **Inteligencia Artificial (IA)** es una rama de la informática que busca crear máquinas capaces de **simular la inteligencia humana**. El objetivo principal es que estas máquinas puedan realizar tareas que normalmente requieren capacidades cognitivas humanas, como aprender, razonar, resolver problemas, percibir, comprender el lenguaje y tomar decisiones.

En esencia, la IA se trata de desarrollar sistemas que pueden:

1.  **Aprender de la experiencia (datos):** En lugar de ser programadas explícitamente para cada tarea, las máquinas con IA pueden analizar grandes volúmenes de datos, identificar patrones y mejorar su rendimiento con el tiempo. Esto es lo que se conoce como **Aprendizaje Automático (Machine Learning)**, que es el motor de la mayoría de las aplicaciones de IA actuales.
2.  **Razonar y resolver problemas:** Utilizar la lógica y la información disponible para llegar a conclusiones o encontrar soluciones a situaciones complejas.
3.  **Percibir el entorno:** Interpretar in

In [8]:
PROMPT_TRIAJE = """
Eres un especialista en triaje del Service Desk para politicas internas.
Dado el mensaje del usuario, devuelve SÓLO un JSON con:\n
{\n
    "decision": "AUTO_RESOLVER" | "PEDIR_INFO" | "ABRIR_TICKET",\n
    "urgency": "BAJA" | "MEDIANA" | "ALTA",\n
    "missing_fields": ["..."]\n
}\n
Reglas:\n
- **AUTO_RESOLVER**: Preguntas claras sobre las reglas o procedimientos descritos en las politicas (Ej.: "¿Puedo reembolsar el internet para mi oficina en casa?").\n
- **PEDIR_INFO**: Mensajes imprecisos o sin información para identificar el tema o el contexto (Ej.: "Necesito ayuda con una politica").\n
- **ABRIR_TICKET**: Solicitudes de excepciones, autorización, aprobación o acceso especial, o cuando el usuario solicita explicitamente abrir un ticket (Ej.: "Quiero una excepción para trabajar remotamente durante 5 dias").\n
Analiza el mensaje y decide la acción más adecuada.
"""

In [9]:
from typing import Literal, List, Dict

In [10]:
from pydantic import BaseModel, Field

In [11]:
class TriajeOut(BaseModel):
    decision: Literal["AUTO_RESOLVER", "PEDIR_INFO", "ABRIR_TICKET"]
    urgencia: Literal["BAJA", "MEDIANA", "ALTA"]
    campos_faltantes: List[str] = Field(default_factory=list)

In [12]:
from langchain_core.messages import SystemMessage, HumanMessage

In [13]:
chain_de_triaje = llm.with_structured_output(TriajeOut)

def triaje(mensaje: str) -> Dict:
    salida: TriajeOut = chain_de_triaje.invoke(
        [
            SystemMessage(content=PROMPT_TRIAJE),
            HumanMessage(content=mensaje)
        ]
    )
    return salida.model_dump()

In [14]:
mensajes_de_prueba = [
    "¿Puedo obtener un reembolso por el internet de mi home office?",
    "Quiero una excepción para teletrabajar durante 5 días.",
    "¿Cómo funciona la política de comidas para viajes?",
    "¿Existe una política para anticipos de vacaciones?",
    "¿Quién fue Napoleón Bonaparte?"
]

In [15]:
for pregunta in mensajes_de_prueba:
    r = triaje(pregunta)
    print(f"{pregunta} -> {r}")

¿Puedo obtener un reembolso por el internet de mi home office? -> {'decision': 'AUTO_RESOLVER', 'urgencia': 'BAJA', 'campos_faltantes': []}
Quiero una excepción para teletrabajar durante 5 días. -> {'decision': 'ABRIR_TICKET', 'urgencia': 'MEDIANA', 'campos_faltantes': []}
¿Cómo funciona la política de comidas para viajes? -> {'decision': 'AUTO_RESOLVER', 'urgencia': 'BAJA', 'campos_faltantes': []}
¿Existe una política para anticipos de vacaciones? -> {'decision': 'AUTO_RESOLVER', 'urgencia': 'BAJA', 'campos_faltantes': []}
¿Quién fue Napoleón Bonaparte? -> {'decision': 'PEDIR_INFO', 'urgencia': 'BAJA', 'campos_faltantes': ['contexto_politica_interna']}


## Describiendo los documentos de recursos humanos

In [16]:
!pip install -q langchain_community faiss-cpu langchain-text-splitters pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [17]:
from pathlib import Path
from langchain_community.document_loaders import PyMuPDFLoader

docs = []

for n in Path("/content/").glob("*.pdf"):
    try:
        loader = PyMuPDFLoader(str(n))
        docs.extend(loader.load())
        print(f"Archivo cargado: {n.name}")
    except Exception as e:
        print(f"Error cargando archivo: {n.name}: {e}")

print(f"Total de documentos cargados: {len(docs)}")

/tmp/ipykernel_4279/1455878096.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


Archivo cargado: Política de Uso de Correo Electrónico y Seguridad de la Información.pdf
Archivo cargado: Política de Reembolsos (Viajes y Gastos).pdf
Archivo cargado: Política de Teletrabajo (Home Office).pdf
Total de documentos cargados: 3


In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [19]:
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)

In [20]:
docs_splits = splitter.split_documents(docs)

In [21]:
for chunk in docs_splits:
    print(chunk)
    print("-----------------")

page_content='Política de Uso de Correo Electrónico 
y Seguridad de la Información 
1. Propósito Esta política define el uso aceptable de los sistemas de correo electrónico de 
la empresa y las responsabilidades de los empleados para proteger la información' metadata={'producer': 'Skia/PDF m143 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': '/content/Política de Uso de Correo Electrónico y Seguridad de la Información.pdf', 'file_path': '/content/Política de Uso de Correo Electrónico y Seguridad de la Información.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': 'Política de Reembolsos (Viajes y Gastos)', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}
-----------------
page_content='corporativa contra accesos no autorizados, pérdida o divulgación. 
2. Uso del Correo Electrónico (E-mail) 
●​ Uso Profesional: La cuenta de correo electrónico es una herramienta de trabajo y 
debe usarse princi

In [22]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [23]:
modelo_embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=GEMINI_API_KEY
)

In [24]:
from langchain_community.vectorstores import FAISS

In [25]:
vectorstore = FAISS.from_documents(docs_splits, modelo_embeddings)

In [26]:
retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.3, "k": 4}
)

In [30]:
# Suponemos que tenemos una función para buscar documentos
def buscar_documentos(pregunta):
    # Esta función devuelve documentos relevantes
    documentos = {
        "pandas": "Los pandas se alimentan principalmente de bambú."
    }
    return documentos.get(pregunta.lower(), "")

# Función para generar respuesta usando GPT
def generar_respuesta(pregunta):
    # Primero, recuperamos documentos relevantes
    documento = buscar_documentos(pregunta)

    # Luego, usamos el modelo de IA para generar una respuesta
    if documento:
        respuesta = f"Basado en información interna, sabemos que {documento}"
    else:
        respuesta = "Lo siento, no tengo suficiente información para responder a eso."

    return respuesta

# Ejemplo de uso
pregunta = "¿Qué comen los pandas?"
print(generar_respuesta(pregunta))

Lo siento, no tengo suficiente información para responder a eso.


In [39]:
from langchain_core.prompts import ChatPromptTemplate
# from langchain.chains.combine_documents import create_stuff_documents_chain # This import is no longer needed and causes a ModuleNotFoundError

In [42]:
prompt_rag = ChatPromptTemplate(
    [
        ("system",
            """Eres el especialista en RR.HH. de la empresa Carraro Desarrollo de Software.
            Responde siempre utilizando los conocimientos de las bases de datos pasadas a ti.
            Si no hay informacion sobre la pregunta en los datos, responde solo 'No lo se'.
            """
        ),
        ("human", "Contexto: {context}\nPregunta del empleado: {question}") # Changed {input} to {question}
    ]
)

# document_chain = create_stuff_documents_chain(llm, prompt_rag) # This is no longer needed and caused an error

In [46]:
def busqueda_de_respuestas_RAG(pregunta) -> Dict:
  # The 'retriever' variable in the global scope will now refer to the one
  # associated with the Chroma vector store (defined in cell LsZXp5N89aAB).
  documentos_relacionados = retriever.invoke(pregunta)

  if not documentos_relacionados:
    return {
        "respuesta": "No lo sé",
        "citaciones": [],
        "documentos_encontrados": False
    }

  # Use the 'rag_chain' (defined in cell 3CKgOUx99tVA) which is functional.
  # It handles the context formatting and LLM invocation internally.
  answer = rag_chain.invoke(pregunta)

  if answer.rstrip(".!?") == "No lo sé":
      return {
          "respuesta": "No lo sé",
          "citaciones": [],
          "documentos_encontrados": False
      }

  return {
      "respuesta": answer,
      "citaciones": documentos_relacionados, # Use the separately retrieved documents for citations
      "documentos_encontrados": True
  }

In [51]:
busqueda_de_respuestas_RAG("¿Puedo obtener un reembolso por el internet de mi home office?")

{'respuesta': 'Sí, los empleados bajo el modelo de trabajo remoto (Home Office) aprobado pueden solicitar un reembolso mensual de hasta 35 EUR para gastos de conexión a internet, presentando la factura correspondiente.',
 'citaciones': [Document(id='dd30c652-7f3c-48be-9daa-65cf50078029', metadata={'producer': 'Skia/PDF m143 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': '/content/Política de Reembolsos (Viajes y Gastos).pdf', 'file_path': '/content/Política de Reembolsos (Viajes y Gastos).pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': 'Política de Reembolsos (Viajes y Gastos)', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}, page_content='con recibo. \n●\u200b Internet (Home Office): Los empleados bajo el modelo de trabajo remoto (Home \nOffice) aprobado (ver Política de Home Office) pueden solicitar un reembolso \nmensual de hasta 35 EUR para gastos de conexión a internet, presentando

In [52]:
mensajes_de_prueba = [
    "¿Puedo obtener un reembolso por el internet de mi home office?",
    "Quiero una excepción para teletrabajar durante 5 días.",
    "¿Cómo funciona la política de comidas para viajes?",
    "¿Existe una politica para anticipos de vacaciones?",
    "¿Quién fue Napoleon Bonaparte?"
]

In [53]:
r = busqueda_de_respuestas_RAG("¿Puedo obtener un reembolso por el internet de mi home office?")
print(r)

{'respuesta': 'Sí, los empleados bajo el modelo de trabajo remoto (Home Office) aprobado pueden solicitar un reembolso mensual de hasta 35 EUR para gastos de conexión a internet, presentando la factura correspondiente.', 'citaciones': [Document(id='dd30c652-7f3c-48be-9daa-65cf50078029', metadata={'producer': 'Skia/PDF m143 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': '/content/Política de Reembolsos (Viajes y Gastos).pdf', 'file_path': '/content/Política de Reembolsos (Viajes y Gastos).pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': 'Política de Reembolsos (Viajes y Gastos)', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}, page_content='con recibo. \n●\u200b Internet (Home Office): Los empleados bajo el modelo de trabajo remoto (Home \nOffice) aprobado (ver Política de Home Office) pueden solicitar un reembolso \nmensual de hasta 35 EUR para gastos de conexión a internet, presentando 

In [54]:
print(f"Número de citaciones para la última respuesta: {len(r['citaciones'])}")

Número de citaciones para la última respuesta: 4


In [ ]:
for pregunta in mensajes_de_prueba:
    respuesta_RAG = busqueda_de_respuestas_RAG(pregunta)

    print(f"\nPREGUNTA: {pregunta}")
    print(f"RESPUESTA: {respuesta_RAG['respuesta']}")
    print(f"DOCUMENTOS ENCONTRADOS: {respuesta_RAG['documentos_encontrados']}")

    if respuesta_RAG['documentos_encontrados']:
        for i, citacion in enumerate(respuesta_RAG['citaciones'], start=1):
            print(f"\nCITACION {i}:")
            print(
                f"Camino del documento: "
                f"{citacion.metadata.get('file_path', 'No disponible')}"
            )
            print(
                f"Contenido: "
                f"{citacion.page_content.replace(chr(10), ' ')}"
            )

    print("\n" + "=" * 80)

In [ ]:
import time

# Nos aseguramos de que el rag_chain use el LLM local para evitar errores de cuota
# hf_llm fue definido en la celda fe500b6a
if 'hf_llm' in globals():
    llm = hf_llm

for pregunta in mensajes_de_prueba:
    try:
        print(f"PROCESANDO: {pregunta}")
        respuesta_RAG = busqueda_de_respuestas_RAG(pregunta)

        print(f"PREGUNTA: {pregunta}")
        print(f"RESPUESTA: {respuesta_RAG['respuesta']}")
        print(f"DOCUMENTOS ENCONTRADOS: {respuesta_RAG['documentos_encontrados']}")

        if respuesta_RAG['documentos_encontrados']:
            for i, citacion in enumerate(respuesta_RAG['citaciones']):
                print(f"  CITACION {i + 1}:")
                # Usamos .get() para evitar errores si la clave no existe
                source = citacion.metadata.get('file_path', citacion.metadata.get('source', 'Desconocido'))
                print(f"    Origen: {source}")
                print(f"    Contenido: {citacion.page_content[:200].replace('\n', ' ')}...")

        print("-" * 30)
        # Pequeña pausa para no saturar la salida si el modelo es muy rápido
        time.sleep(0.5)

    except Exception as e:
        print(f"Error al procesar la pregunta '{pregunta}': {e}")

# OTRO PLANTEAMIENTO

## Instalar dependencias

In [ ]:
# !pip install -q \
# langchain \
# langchain-openai \
# langchain-community \
# langchain-core \
# chromadb \
# pypdf \
# tiktoken \
# langchain-text-splitters # Commented out to prevent dependency conflicts

 ## Configurar API Key

In [ ]:
from google.colab import userdata
import os

os.environ["OPEN_AI_API_KEY"] = userdata.get("OPEN_AI_API_KEY")

## Subir PDFs

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving Política de Uso de Correo Electrónico y Seguridad de la Información.pdf to Política de Uso de Correo Electrónico y Seguridad de la Información.pdf
Saving Política de Teletrabajo (Home Office).pdf to Política de Teletrabajo (Home Office).pdf
Saving Política de Reembolsos (Viajes y Gastos).pdf to Política de Reembolsos (Viajes y Gastos).pdf


## Cargar documentos

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

documents = []

for file_name in uploaded.keys():
    loader = PyPDFLoader(file_name)
    docs = loader.load()

    documents.extend(docs)

print(f"Documentos cargados: {len(documents)}")

/tmp/ipykernel_6060/1093662198.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Documentos cargados: 3


## Chunking

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print(f"Chunks creados: {len(chunks)}")

Chunks creados: 11


## Crear embeddings

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

documents = []

for file_name in uploaded.keys():
    loader = PyPDFLoader(file_name)
    docs = loader.load()

    documents.extend(docs)

print(f"Documentos cargados: {len(documents)}")

Documentos cargados: 3


In [ ]:
from langchain_openai import OpenAIEmbeddings
from google.colab import userdata
from google.colab.userdata import SecretNotFoundError

try:
    # Attempt to retrieve the API key directly from Colab secrets
    # The context indicates the user might have set 'OPEN_AI_API_KEY' or 'openai_key'.
    # This code specifically looks for 'OPEN_AI_API_KEY'.
    openai_api_key = userdata.get("OPEN_AI_API_KEY")
except SecretNotFoundError:
    raise ValueError(
        "OpenAI API key not found. Please ensure you have set a Colab secret "
        "named 'OPEN_AI_API_KEY' (or 'openai_key' if that's what you intended to use, "
        "and adjust the code accordingly). Alternatively, set the 'OPENAI_API_KEY' "
        "environment variable (without the underscore in 'OPENAI_API_KEY' for LangChain)."
    )

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=openai_api_key  # Pass the key directly to the constructor
)

### Usar Embeddings de HuggingFace (gratuitos y locales)

In [ ]:
# Instalar las librerías necesarias para HuggingFace Embeddings y la versión recomendada por LangChain
# !pip install -q --force-reinstall langchain-huggingface sentence-transformers transformers # Commented out to prevent dependency conflicts

### Instalar librerías adicionales para ejecutar modelos locales (HuggingFace LLM)

In [ ]:
# `accelerate` y `bitsandbytes` son útiles para ejecutar modelos más grandes de forma eficiente.
# `torch` debería estar instalado por otras dependencias, pero asegúrate de que sea una versión compatible.
# !pip install -q accelerate bitsandbytes # Commented out to prevent dependency conflicts

In [ ]:
# Importar HuggingFaceEmbeddings de la nueva ubicación recomendada
from langchain_huggingface import HuggingFaceEmbeddings

# Inicializar HuggingFaceEmbeddings con un modelo pre-entrenado. Puedes elegir otros modelos si lo deseas.
# 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2' es un buen modelo multilingüe.
# Para modelos en español, considera 'hiiamsid/sentence_similarity_spanish_es'
# O 'distilbert-base-nli-stsb-mean-tokens' para inglés (requiere descargar el modelo)

hf_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# Asignar la instancia de HuggingFaceEmbeddings a la variable 'embeddings'
# para que el siguiente paso de Chroma.from_documents la use.
embeddings = hf_embeddings

print("Embeddings de HuggingFace inicializados correctamente.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  except userdata.SecretNotFoundError:


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings de HuggingFace inicializados correctamente.


In [ ]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

print("Base vectorial creada")

Base vectorial creada


In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

In [ ]:
query = "¿Cuál es el objetivo principal del documento?"

results = retriever.invoke(query)

for i, doc in enumerate(results):
    print(f"\n--- Chunk {i+1} ---\n")
    print(doc.page_content[:500])


--- Chunk 1 ---

Política  de  Uso  de  Correo  Electrónico  
y
 
Seguridad
 
de
 
la
 
Información
 
1.  Propósito  Esta  política  define  el  uso  aceptable  de  los  sistemas  de  correo  electrónico  de  
la
 
empresa
 
y
 
las
 
responsabilidades
 
de
 
los
 
empleados
 
para
 
proteger
 
la
 
información
 
corporativa
 
contra
 
accesos
 
no
 
autorizados,
 
pérdida
 
o
 
divulgación.
 
2.  Uso  del  Correo  Electrónico  (E-mail)  
●  Uso  Profesional:  La  cuenta  de  correo  electrónico  es  una  herram

--- Chunk 2 ---

Política  de  Reembolsos  (Viajes  y  
Gastos)
 
1.  Objetivo  Establecer  las  directrices  y  procedimientos  para  el  reembolso  de  gastos  
incurridos
 
por
 
los
 
empleados
 
en
 
el
 
ejercicio
 
de
 
sus
 
funciones
 
oficiales,
 
asegurando
 
la
 
transparencia,
 
equidad
 
y
 
cumplimiento
 
fiscal.
 
2.  Ámbito  de  Aplicación  Esta  política  se  aplica  a  todos  los  empleados  fijos  y  temporales  
que
 
incurran
 
en
 
gastos
 
en
 
nombre


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Eres un asistente especializado en responder preguntas
usando exclusivamente el contexto proporcionado.

Reglas:

1. Responde únicamente con información presente en el contexto.
2. No inventes datos.
3. Si la respuesta no está en el contexto, responde:

"No encontré información suficiente en los documentos para responder esa pregunta."

Contexto:

{context}

Pregunta:

{question}

Respuesta:
""")

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    api_key=openai_api_key # Pasa la clave directamente al constructor
)

### Configurar un LLM de Hugging Face

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline
import torch

# Define el nombre del modelo de Hugging Face que quieres usar
# TinyLlama/TinyLlama-1.1B-Chat-v1.0 es un modelo pequeño adecuado para pruebas en Colab
# Si tienes GPU y más RAM, puedes probar modelos más grandes como 'google/gemma-2b-it'
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Cargar el tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Cargar el modelo. Usa torch_dtype=torch.bfloat16 para mayor eficiencia si tu GPU lo soporta.
# Si tienes problemas de memoria, puedes intentar cargar con quantization_config (e.g., bitsandbytes)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto")

# Crear un pipeline de texto con el modelo y el tokenizer
pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=100,
    temperature=0.1,
    do_sample=False
    # max_new_tokens=512, # Limita la longitud de la respuesta generada
    # temperature=0.1,    # Ajusta para controlar la creatividad de la respuesta
)

# Inicializar el LLM de HuggingFace para LangChain
hf_llm = HuggingFacePipeline(pipeline=pipeline)

# Sobrescribir la variable `llm` para que el `rag_chain` use el nuevo LLM de Hugging Face
llm = hf_llm

print(f"LLM de Hugging Face '{model_name}' inicializado correctamente y configurado para la cadena RAG.")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM de Hugging Face 'TinyLlama/TinyLlama-1.1B-Chat-v1.0' inicializado correctamente y configurado para la cadena RAG.


In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [49]:
# The format_docs function definition was moved to cell 3CKgOUx99tVA to ensure it's defined before rag_chain.

In [50]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt_rag # Changed prompt to prompt_rag
    | llm  # This will now correctly reference the HuggingFace LLM
    | StrOutputParser()
)

In [ ]:
docs = retriever.invoke(
    "¿Cada cuánto deben cambiarse las contraseñas?"
)

print(format_docs(docs))

alta
 
velocidad
 
estable.
 
6.  Reembolsos  Asociados  Como  se  detalla  en  la  "Política  de  Reembolsos",  los  empleados  
en
 
esta
 
modalidad
 
pueden
 
solicitar
 
el
 
reembolso
 
de
 
gastos
 
de
 
internet
 
hasta
 
el
 
límite
 
establecido
 
(35
 
EUR/mes).
 
La
 
empresa
 
no
 
reembolsará
 
gastos
 
de
 
mobiliario
 
(sillas,
 
mesas)
 
ni
 
electricidad.
 
7.  Solicitudes  de  Excepción  (Full  Remote)  El  modelo  híbrido  (3+2)  es  la  norma.  Las  
solicitudes
 
para
 
un
 
modelo
 
100%
 
remoto
 
(ej.
 
trabajar
 
5
 
días
 
desde
 
casa
 
o
 
desde
 
otra
 
ciudad)
 
se
 
consideran
 
excepciones.
 
Estas
 
solicitudes
 
deben
 
presentarse
 
por
 
escrito
 
al
 
gerente
 
y
 
al
 
departamento
 
de
 
RRHH,
 
detallando
 
la
 
justificación.
 
Serán
 
evaluadas
 
caso
 
por
 
caso
 
por
 
la
 
Dirección,
 
basándose
 
en
 
el
 
rendimiento
 
y
 
la
 
naturaleza
 
del
 
puesto.

cada
 
90
 
días.
 
4.  Manejo  de  Datos  Confidenciales  Toda  información  sensi

In [ ]:
rag_chain.invoke(
    "¿Cuál es la política de licencia por maternidad?"
)

[transformers] Both `max_new_tokens` (=512) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] This is a friendly reminder - the current text generation call has exceeded the model's predefined maximum length (2048). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.


'Human: \nEres un asistente especializado en responder preguntas\nusando exclusivamente el contexto proporcionado.\n\nReglas:\n\n1. Responde únicamente con información presente en el contexto.\n2. No inventes datos.\n3. Si la respuesta no está en el contexto, responde:\n\n"No encontré información suficiente en los documentos para responder esa pregunta."\n\nContexto:\n\nReservas\n \nfuera\n \nde\n \neste\n \nlímite\n \nrequieren\n \njustificación\n \ny\n \naprobación.\n ●  Transporte  Terrestre:  Se  reembolsarán  taxis,  servicios  de  VTC  (Uber/Cabify)  o  \ntransporte\n \npúblico\n \nnecesarios\n \npara\n \ndesplazamientos\n \nlaborales\n \n(aeropuertos,\n \nreuniones).\n \nEl\n \nuso\n \nde\n \nvehículo\n \npersonal\n \nserá\n \nreembolsado\n \npor\n \nkilómetro\n \nsegún\n \nla\n \ntarifa\n \noficial\n \nvigente.\n \n4.  Gastos  de  Manutención  (Alimentación)  \n●  Viajes  Nacionales/Internacionales:  Se  establece  una  dieta  diaria  fija  (viático)  de  70  \nEUR\n \npara\n \